In [14]:
# проверка текстов на наличие артефактов выгрузки и других признаков

import pandas as pd


CHECKS = {
    # артефакты выгрузки
    "HTML-теги":                r"<[a-zA-Z/][^>]*>",
    "HTML-сущности":            r"&(?:quot|amp|lt|gt|nbsp|apos|#\d+);",
    "невидимые символы":        r"[\u00ad\u2060\u202a-\u202e\u200b\u200c\u200e\u200f\ufeff]",
    "переносы строк":           r"\n",

    "обращение":                r"^\s*@\w",
    "блок кода":                r"```",
    "команда бота":             r"^\s*/\w+",
    "цитата в начале":          r"^\s*>",
    "отсутствие кириллицы":                r"^[^А-Яа-яЁё]*$",
    "другой кириллический":     r"[іїєґІЇЄҐўЎђћјљњЂЋЈЉЊ]",

    "ссылка":                   r"https?://|www\.",
    "обращение в середине":     r"@\w",
    "хэштег":                   r"#\w",
    "эмодзи":                   "[\U0001F300-\U0001FAFF\u2600-\u27BF]",
    "повтор буквы 5+ раз":      r"([А-Яа-яЁё])\1{4,}"
}

FILES = {
    "YouTube": "./data/raw/yt-comments.parquet",
    "Telegram": "./data/raw/tg-groups.parquet",
    "RuReviews": "./data/raw/ru-reviews.parquet",
}

rows = []
for name, path in FILES.items():
    text = pd.read_parquet(path).text.astype(str).astype(object)
    row = {"корпус": name, "документов": len(text)}
    for label, pattern in CHECKS.items():
        n = text.str.contains(pattern, regex=True, na=False).sum()
        row[label] = f"{n} ({n / len(text):.1%})"
    rows.append(row)

print(pd.DataFrame(rows).set_index("корпус").T.to_string())

/tmp/ipykernel_1798/1074512168.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n = text.str.contains(pattern, regex=True, na=False).sum()
/tmp/ipykernel_1798/1074512168.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n = text.str.contains(pattern, regex=True, na=False).sum()
/tmp/ipykernel_1798/1074512168.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n = text.str.contains(pattern, regex=True, na=False).sum()


корпус                     YouTube      Telegram    RuReviews
документов                   53969        359525        90000
HTML-теги                19 (0.0%)   1036 (0.3%)     0 (0.0%)
HTML-сущности             0 (0.0%)     14 (0.0%)     0 (0.0%)
невидимые символы         0 (0.0%)    154 (0.0%)    51 (0.1%)
переносы строк            0 (0.0%)  28189 (7.8%)  4661 (5.2%)
обращение                 2 (0.0%)   1794 (0.5%)     0 (0.0%)
блок кода                 1 (0.0%)     64 (0.0%)     0 (0.0%)
команда бота              4 (0.0%)    279 (0.1%)     1 (0.0%)
цитата в начале          12 (0.0%)    695 (0.2%)     0 (0.0%)
отсутствие кириллицы   1273 (2.4%)   9005 (2.5%)  1262 (1.4%)
другой кириллический    261 (0.5%)    128 (0.0%)     2 (0.0%)
ссылка                   97 (0.2%)   2454 (0.7%)   128 (0.1%)
обращение в середине     64 (0.1%)   3933 (1.1%)    22 (0.0%)
хэштег                   36 (0.1%)   1690 (0.5%)    61 (0.1%)
эмодзи                8670 (16.1%)   7703 (2.1%)   116 (0.1%)
повтор б

In [15]:
import re
import html
import unicodedata
import numpy as np
import pandas as pd

WORD    = re.compile(r"[А-Яа-яЁё]+")
FOREIGN = re.compile(r"[іїєґІЇЄҐўЎђћјљњЂЋЈЉЊ]")
KEY     = re.compile(r"[^а-яё]")

TAG_A   = re.compile(r"<a\b[^>]*>(.*?)</a\s*>", re.I | re.S)
TAG_ANY = re.compile(r"</?[a-zA-Z][a-zA-Z0-9]{0,9}(?:\s[^<>]{0,200})?/?>")
INVISIBLE   = re.compile("[\u00ad\u2060\u202a-\u202e\u200b\u200c\u200e\u200f\ufeff]")


def n_words(text):
    return len(WORD.findall(str(text)))


def fix_export(text):
    text = str(text)
    for _ in range(3):
        before = text
        text = TAG_A.sub(r"\1", text)
        text = TAG_ANY.sub(" ", text)
        text = html.unescape(text)
        if text == before:
            break
    text = INVISIBLE.sub("", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Cc" or c == "\n")
    text = re.sub(r"\n{2,}", "\n", text)
    return re.sub(r"[ \t]+", " ", text).strip()

In [16]:
LO, HI = 3, 35


def apply_filters(df, name):
    
    print(name)
    print(f"вход: {len(df)}")

    d = df[["text"]].copy()

    # удаление артефактов выгрузки
    d["text"] = d.text.astype(str).map(fix_export)

    was = len(d)
    d = d[d.text.str.strip() != ""]
    print(f"непустые: {len(d)} (-{was - len(d)})")

    was = len(d)
    d = d[d.text.map(n_words) > 0]
    print(f"с кириллицей: {len(d)} (-{was - len(d)})")

    was = len(d)
    d = d[~d.text.str.contains(FOREIGN, regex=True)]
    print(f"русские кириллические: {len(d)} (-{was - len(d)})")

    was = len(d)
    d = d.drop_duplicates(subset="text")
    print(f"без точных дублей: {len(d)} (-{was - len(d)})")

    # различаются только пунктуацией и регистром
    was = len(d)
    key = d.text.str.lower().str.replace(KEY, "", regex=True)
    d = d[~key.duplicated()]
    print(f"без дублей по ключу: {len(d)} (-{was - len(d)})")

    was = len(d)
    d = d[d.text.map(n_words).between(LO, HI)]
    print(f"в диапазоне {LO}-{HI}: {len(d)} (-{was - len(d)})")

    print(f"осталось: {len(d) / len(df):.0%}")
    d = d.reset_index(drop=True)
    d.insert(0, "id", range(1, len(d) + 1))
    return d

In [17]:
yt = apply_filters(pd.read_parquet("./data/raw/yt-comments.parquet"), "YouTube")
tg = apply_filters(pd.read_parquet("./data/raw/tg-groups.parquet"), "Telegram")
rr = apply_filters(pd.read_parquet("./data/raw/ru-reviews.parquet"), "RuReviews")

YouTube
вход: 53969
непустые: 53969 (-0)
с кириллицей: 52696 (-1273)
русские кириллические: 52435 (-261)
без точных дублей: 51776 (-659)
без дублей по ключу: 50979 (-797)
в диапазоне 3-35: 37537 (-13442)
осталось: 70%
Telegram
вход: 359525
непустые: 359489 (-36)
с кириллицей: 350515 (-8974)
русские кириллические: 350390 (-125)
без точных дублей: 350375 (-15)
без дублей по ключу: 338990 (-11385)
в диапазоне 3-35: 298738 (-40252)
осталось: 83%
RuReviews
вход: 90000
непустые: 89999 (-1)
с кириллицей: 88738 (-1261)
русские кириллические: 88736 (-2)
без точных дублей: 85927 (-2809)
без дублей по ключу: 83913 (-2014)
в диапазоне 3-35: 69976 (-13937)
осталось: 78%


In [18]:
N_WORK = 15_000   # рабочая выборка
N_REF = 20_000    # референс

rows = []

for name, d in [("YouTube", yt), ("Telegram", tg), ("RuReviews", rr)]:
    ref = len(d) - N_WORK
    rows.append({"корпус": name, "после фильтров": len(d), "рабочая": min(N_WORK, len(d)), "референс": max(0, ref), "достаточность": 'да' if ref >= N_REF else 'нет'})

print(pd.DataFrame(rows).to_string(index=False))

   корпус  после фильтров  рабочая  референс достаточность
  YouTube           37537    15000     22537            да
 Telegram          298738    15000    283738            да
RuReviews           69976    15000     54976            да


In [19]:
rows = []
for name, d in [("YouTube", yt), ("Telegram", tg), ("RuReviews", rr)]:
    nw = d.text.map(n_words)
    rows.append({"корпус": name, "документов": len(d), "медиана": int(nw.median()), "среднее": round(nw.mean(), 1), "25%": int(np.percentile(nw, 25)), "75%": int(np.percentile(nw, 75)), "90%": int(np.percentile(nw, 90))})

print(pd.DataFrame(rows).to_string(index=False))

   корпус  документов  медиана  среднее  25%  75%  90%
  YouTube       37537       11     13.1    6   18   26
 Telegram      298738        8      9.6    5   12   19
RuReviews       69976       13     14.5    8   20   27


In [20]:
yt.to_parquet("./data/processed/yt-comments.parquet", index=False)
tg.to_parquet("./data/processed/tg-groups.parquet", index=False)
rr.to_parquet("./data/processed/ru-reviews.parquet", index=False)